# CONTRACT REVIEW & REDLINING AI AGENT PIPELINE  | ║   LegalTech SME Automation | CUAD + LLM Fine-Tuning

In [1]:
# Install dependencies
!pip install transformers accelerate peft bitsandbytes datasets pdfplumber
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install PyPDF2 python-docx openpyxl matplotlib seaborn scikit-learn
!pip install tqdm psutil

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 107.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 102.9 MB/s eta 0:00:00
Looking in indexes: https://download.pytorch.org/whl/cu118
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 20.1 MB/s eta 0:00:00


In [2]:
#!/usr/bin/env python3
"""
=============================================================================
  CONTRACT REVIEW & REDLINING AI AGENT — FULL ALGORITHMIC PIPELINE
  LegalTech SME Automation | CUAD Dataset | Qwen3 + LLaMA Fine-Tuning
  March 2026 Updated | Production-Grade Implementation
  ** RESOURCE‑AWARE & COMPATIBLE EDITION ** – Runs on GPU 8GB / RAM 20GB
=============================================================================
"""

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 0: IMPORTS & ENVIRONMENT CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
import os
import sys
import gc
import json
import time
import math
import copy
import logging
import warnings
import traceback
import subprocess
import shutil
from pathlib import Path
from datetime import datetime
from collections import defaultdict, Counter
from typing import Dict, List, Optional, Tuple, Any, Union

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:512"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import matplotlib
matplotlib.use("Agg")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm import tqdm

import psutil
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim.lr_scheduler import ReduceLROnPlateau

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_auc_score
)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.utils.class_weight import compute_class_weight
import scipy.stats as stats

from transformers import (
    AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding,
    EarlyStoppingCallback, get_linear_schedule_with_warmup,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from datasets import Dataset as HFDataset
import huggingface_hub
from huggingface_hub import login

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 1: GLOBAL CONFIGURATION (REDUCED MEMORY FOOTPRINT)
# ─────────────────────────────────────────────────────────────────────────────
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

CFG = {
    "root_path"       : "/kaggle/input/datasets/tobimichigan/cuad-contract-understanding-atticus-dataset/",
    "cuad_path"       : "/kaggle/input/datasets/tobimichigan/cuad-contract-understanding-atticus-dataset/CUADv1.json",
    "test_path"       : "/kaggle/input/datasets/tobimichigan/cuad-contract-understanding-atticus-dataset/test.json",
    "train_path"      : "/kaggle/input/datasets/tobimichigan/cuad-contract-understanding-atticus-dataset/train_separate_questions.json",
    "hf_token_path"   : "/kaggle/input/datasets/tobimichigan/hf-tokens/hf_tokens/hf.token.txt",
    "output_dir"      : "./contract_agent_outputs",
    "plots_dir"       : "./contract_agent_outputs/plots",
    "models_dir"      : "./contract_agent_outputs/models",
    "finetuned_dir"   : "./contract_agent_outputs/finetuned_model",

    "primary_model"   : "Qwen/Qwen2.5-7B-Instruct",
    "backup_model"    : "meta-llama/Llama-3.1-8B-Instruct",
    "embed_model"     : "sentence-transformers/all-MiniLM-L6-v2",

    "train_ratio"     : 0.40,
    "val_ratio"       : 0.15,
    "test_ratio"      : 0.15,
    "holdout_ratio"   : 0.30,

    "max_length"      : 256,
    "batch_size"      : 4,
    "grad_accum"      : 2,
    "epochs"          : 8,
    "lr"              : 2e-4,
    "weight_decay"    : 0.01,
    "warmup_ratio"    : 0.1,
    "dropout"         : 0.3,
    "early_stop_patience": 3,
    "lora_r"          : 8,
    "lora_alpha"      : 16,
    "lora_dropout"    : 0.05,

    "memory_limit_gb" : 0.85,
    "chunk_size"      : 200,
    "max_samples"     : 1000,

    "seed"            : 42,
}

CUAD_CATEGORIES = [
    "Document Name", "Parties", "Agreement Date", "Effective Date",
    "Expiration Date", "Renewal Term", "Notice Period To Terminate Renewal",
    "Governing Law", "Termination For Convenience", "Change Of Control",
    "Anti-Assignment", "Non-Compete", "Exclusivity", "Liability Cap",
    "Liquidated Damages", "IP Ownership Assignment", "Warranty Duration",
    "Audit Rights", "Most Favored Nation", "Confidentiality Duration",
]

RISK_LEVELS = {
    "HIGH"  : ["Liability Cap", "Termination For Convenience", "Change Of Control",
                "IP Ownership Assignment", "Anti-Assignment"],
    "MEDIUM": ["Non-Compete", "Exclusivity", "Governing Law", "Renewal Term"],
    "LOW"   : ["Document Name", "Parties", "Agreement Date", "Expiration Date"],
}

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 2: LOGGING & UTILITY HELPERS
# ─────────────────────────────────────────────────────────────────────────────
for d in [CFG["output_dir"], CFG["plots_dir"], CFG["models_dir"], CFG["finetuned_dir"]]:
    Path(d).mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(f"{CFG['output_dir']}/pipeline_{TIMESTAMP}.log"),
        logging.StreamHandler(sys.stdout),
    ]
)
logger = logging.getLogger("ContractAI")

def set_seed(seed: int = 42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    logger.info(f"Seed set to {seed}")

def get_memory_gb() -> float:
    proc = psutil.Process(os.getpid())
    return proc.memory_info().rss / (1024 ** 3)

def get_system_memory() -> Tuple[float, float]:
    vm = psutil.virtual_memory()
    return vm.total / (1024**3), vm.available / (1024**3)

def memory_safe() -> bool:
    total, avail = get_system_memory()
    used_frac = (total - avail) / total
    return used_frac < CFG["memory_limit_gb"]

def force_cleanup(*args):
    for obj in args:
        try:
            del obj
        except Exception:
            pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    logger.debug(f"[MEM] After cleanup: {get_memory_gb():.2f} GB RSS")

def log_memory(tag: str = ""):
    cur = get_memory_gb()
    total, avail = get_system_memory()
    logger.info(f"[MEM {tag}] Process={cur:.2f}GB | Available={avail:.1f}GB / {total:.1f}GB")

def save_plot(fig, name: str, dpi: int = 150):
    path = f"{CFG['plots_dir']}/{name}_{TIMESTAMP}.png"
    fig.savefig(path, dpi=dpi, bbox_inches="tight", facecolor="white")
    logger.info(f"Plot saved → {path}")
    plt.close(fig)
    gc.collect()
    return path

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3: HUGGINGFACE AUTHENTICATION
# ─────────────────────────────────────────────────────────────────────────────
def authenticate_huggingface() -> bool:
    logger.info("=" * 60)
    logger.info("STEP: HuggingFace Authentication")
    logger.info("=" * 60)
    token_path = CFG["hf_token_path"]
    try:
        if Path(token_path).exists():
            with open(token_path, "r") as f:
                token = f.read().strip()
            login(token=token, add_to_git_credential=False)
            logger.info(f" HuggingFace login successful (token from {token_path})")
            return True
        else:
            env_token = os.environ.get("HF_TOKEN", "")
            if env_token:
                login(token=env_token, add_to_git_credential=False)
                logger.info(" HuggingFace login via env HF_TOKEN")
                return True
            logger.warning(f"⚠ Token file not found: {token_path}. Proceeding unauthenticated.")
            return False
    except Exception as e:
        logger.error(f"✗ HuggingFace auth error: {e}")
        return False

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 4: DATA LOADING (CHUNKED, MEMORY-SAFE)
# ─────────────────────────────────────────────────────────────────────────────
def load_cuad_json_chunked(json_path: str, max_samples: int = CFG["max_samples"],
                            chunk_size: int = CFG["chunk_size"]) -> pd.DataFrame:
    logger.info(f"Loading CUAD JSON: {json_path}")
    if not Path(json_path).exists():
        logger.error(f"File not found: {json_path}")
        return pd.DataFrame()

    log_memory("before_load")
    records = []

    with open(json_path, "r", encoding="utf-8") as f:
        raw = json.load(f)

    data_items = raw.get("data", [])
    logger.info(f"  Total contracts in file: {len(data_items)}")

    with tqdm(total=min(len(data_items), max_samples),
              desc=f"  Parsing {Path(json_path).name}", unit="contract") as pbar:

        for doc in data_items:
            if len(records) >= max_samples:
                break
            if not memory_safe():
                logger.warning("  ⚠ Memory limit reached – stopping load early")
                break

            title = doc.get("title", "UNKNOWN")
            paragraphs = doc.get("paragraphs", [])

            for para in paragraphs:
                context = para.get("context", "")[:3000]
                qas = para.get("qas", [])

                for qa in qas:
                    q_id     = qa.get("id", "")
                    question = qa.get("question", "")
                    is_imp   = qa.get("is_impossible", True)
                    answers  = qa.get("answers", [])

                    category = "Unknown"
                    for cat in CUAD_CATEGORIES:
                        if cat.lower().replace("/", "_") in q_id.lower().replace("/", "_"):
                            category = cat
                            break

                    ans_texts = [a.get("text", "").strip() for a in answers if a.get("text")]
                    answer_text = " | ".join(ans_texts[:3]) if ans_texts else ""
                    has_answer  = int(not is_imp and len(ans_texts) > 0)

                    risk = "LOW"
                    for level, cats in RISK_LEVELS.items():
                        if category in cats:
                            risk = level
                            break

                    records.append({
                        "doc_title"     : title,
                        "qa_id"         : q_id,
                        "category"      : category,
                        "question"      : question,
                        "context"       : context,
                        "answer"        : answer_text,
                        "has_answer"    : has_answer,
                        "is_impossible" : int(is_imp),
                        "risk_level"    : risk,
                        "context_len"   : len(context),
                        "answer_len"    : len(answer_text),
                        "num_answers"   : len(ans_texts),
                    })

                    if len(records) >= max_samples:
                        break
                if len(records) >= max_samples:
                    break

            pbar.update(1)
            if len(records) % (chunk_size * 2) == 0:
                gc.collect()

    force_cleanup(raw, data_items)
    df = pd.DataFrame(records)
    log_memory("after_load")
    logger.info(f"   Loaded {len(df):,} QA records from {Path(json_path).name}")
    return df


def load_all_datasets() -> Dict[str, pd.DataFrame]:
    logger.info("=" * 60)
    logger.info("STEP 1: DATA LOADING (chunked, memory-safe)")
    logger.info("=" * 60)

    dfs = {}
    for name, path in [("cuad", CFG["cuad_path"]),
                        ("test", CFG["test_path"]),
                        ("train", CFG["train_path"])]:
        dfs[name] = load_cuad_json_chunked(path, max_samples=CFG["max_samples"])
        log_memory(f"after_{name}")
        gc.collect()

    parts = [df for df in dfs.values() if not df.empty]
    master = pd.concat(parts, ignore_index=True).drop_duplicates(subset=["qa_id"])
    logger.info(f"  Master dataset: {len(master):,} unique QA pairs")
    force_cleanup(*parts)
    return master, dfs

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 5: DATA PREPROCESSING & CLEANING
# ─────────────────────────────────────────────────────────────────────────────
def preprocess_and_clean(df: pd.DataFrame) -> pd.DataFrame:
    logger.info("=" * 60)
    logger.info("STEP 2: PREPROCESSING & CLEANING")
    logger.info("=" * 60)
    n0 = len(df)

    df = df.drop_duplicates(subset=["qa_id"]).copy()
    logger.info(f"  After dedup: {len(df):,} (removed {n0 - len(df):,})")

    with tqdm(df.columns, desc="  Filling NaNs") as cols:
        for col in cols:
            if df[col].dtype == object:
                df[col] = df[col].fillna("").astype(str).str.strip()
            else:
                df[col] = df[col].fillna(0)

    before = len(df)
    df = df[df["context"].str.len() > 20].copy()
    logger.info(f"  Removed {before - len(df):,} rows with empty context")

    with tqdm(["question", "context", "answer"], desc="  Normalising text") as cols:
        for col in cols:
            df[col] = (df[col]
                       .str.replace(r"\s+", " ", regex=True)
                       .str.replace(r"[^\x00-\x7F]", " ", regex=True)
                       .str.strip())

    le_risk = LabelEncoder()
    df["risk_label"] = le_risk.fit_transform(df["risk_level"])

    le_cat = LabelEncoder()
    df["category_label"] = le_cat.fit_transform(df["category"])

    p99 = df["context_len"].quantile(0.99)
    df["context_len"] = df["context_len"].clip(upper=p99)

    numeric_cols = ["context_len", "answer_len", "num_answers"]
    with tqdm(numeric_cols, desc="  Outlier detection") as cols:
        for col in cols:
            z = np.abs(stats.zscore(df[col].fillna(0)))
            outliers = (z > 3).sum()
            if outliers > 0:
                median_val = df[col].median()
                df.loc[z > 3, col] = median_val
                logger.info(f"    {col}: replaced {outliers} outliers with median={median_val:.1f}")

    with tqdm(df.columns, desc="  Downcasting dtypes") as cols:
        for col in cols:
            if df[col].dtype == np.float64:
                df[col] = df[col].astype(np.float32)
            elif df[col].dtype == np.int64:
                df[col] = df[col].astype(np.int32)

    logger.info(f"   Clean dataset: {len(df):,} rows | {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
    log_memory("after_preprocess")
    return df, le_risk, le_cat

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 6: EXPLORATORY DATA ANALYSIS & PLOTS
# ─────────────────────────────────────────────────────────────────────────────
def run_eda(df: pd.DataFrame):
    logger.info("=" * 60)
    logger.info("STEP 3: EXPLORATORY DATA ANALYSIS")
    logger.info("=" * 60)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle("CUAD Dataset — Class Distributions", fontsize=14, fontweight="bold")

    risk_counts = df["risk_level"].value_counts()
    axes[0].bar(risk_counts.index, risk_counts.values,
                color=["#d32f2f","#f57c00","#388e3c"])
    axes[0].set_title("Risk Level Distribution")
    axes[0].set_ylabel("Count")
    for i, v in enumerate(risk_counts.values):
        axes[0].text(i, v + 5, str(v), ha="center", fontsize=9)

    ha_counts = df["has_answer"].value_counts()
    axes[1].bar(["No Answer", "Has Answer"], ha_counts.values, color=["#e57373","#81c784"])
    axes[1].set_title("Answer Availability")
    axes[1].set_ylabel("Count")

    top_cats = df["category"].value_counts().head(15)
    axes[2].barh(top_cats.index[::-1], top_cats.values[::-1], color="#5c6bc0")
    axes[2].set_title("Top 15 Clause Categories")
    axes[2].set_xlabel("Count")

    plt.tight_layout()
    save_plot(fig, "01_class_distributions")

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("Text Length Analysis", fontsize=14, fontweight="bold")

    axes[0].hist(df["context_len"], bins=50, color="#42a5f5", edgecolor="white", alpha=0.8)
    axes[0].axvline(df["context_len"].mean(), color="red", linestyle="--",
                    label=f"Mean={df['context_len'].mean():.0f}")
    axes[0].set_title("Context Length Distribution")
    axes[0].set_xlabel("Characters")
    axes[0].legend()

    axes[1].hist(df["answer_len"].clip(upper=500), bins=50,
                 color="#66bb6a", edgecolor="white", alpha=0.8)
    axes[1].axvline(df["answer_len"].mean(), color="red", linestyle="--",
                    label=f"Mean={df['answer_len'].mean():.0f}")
    axes[1].set_title("Answer Length Distribution")
    axes[1].set_xlabel("Characters")
    axes[1].legend()

    plt.tight_layout()
    save_plot(fig, "02_text_lengths")

    pivot = df.groupby(["category", "risk_level"])["has_answer"].mean().unstack(fill_value=0)
    fig, ax = plt.subplots(figsize=(10, 12))
    sns.heatmap(pivot, annot=True, fmt=".2f", cmap="YlOrRd", ax=ax, linewidths=0.5)
    ax.set_title("Answer Rate by Category × Risk Level", fontsize=13, fontweight="bold")
    plt.tight_layout()
    save_plot(fig, "03_category_risk_heatmap")

    num_df = df[["context_len","answer_len","num_answers","has_answer",
                 "risk_label","category_label","is_impossible"]].copy()
    corr = num_df.corr()
    fig, ax = plt.subplots(figsize=(9, 7))
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
                center=0, ax=ax, linewidths=0.5)
    ax.set_title("Feature Correlation Matrix", fontsize=13, fontweight="bold")
    plt.tight_layout()
    save_plot(fig, "04_correlation_matrix")

    logger.info("   EDA plots saved to plots directory")
    gc.collect()

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 7: FEATURE ENGINEERING
# ─────────────────────────────────────────────────────────────────────────────
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    logger.info("=" * 60)
    logger.info("STEP 4: FEATURE ENGINEERING")
    logger.info("=" * 60)

    with tqdm(total=10, desc="  Engineering features") as pbar:
        df["q_is_extraction"] = df["question"].str.contains(
            "Highlight|extract|identify", case=False, na=False).astype(np.int8)
        pbar.update(1)

        df["context_word_count"] = df["context"].apply(
            lambda x: len(x.split()) if x else 0).astype(np.int32)
        df["context_density"] = (
            df["context_word_count"] / (df["context_len"] + 1e-5)).astype(np.float32)
        pbar.update(1)

        for level in ["HIGH", "MEDIUM", "LOW"]:
            cats = RISK_LEVELS[level]
            df[f"is_risk_{level.lower()}"] = df["category"].isin(cats).astype(np.int8)
        pbar.update(1)

        money_pattern = r"\$|USD|liability|cap|damages|indemnif"
        df["has_money_terms"] = df["context"].str.contains(
            money_pattern, case=False, na=False, regex=True).astype(np.int8)
        pbar.update(1)

        date_pattern = r"\d{1,2}/\d{1,2}/\d{2,4}|January|February|March|April|May|June|July|August|September|October|November|December"
        df["has_date_terms"] = df["context"].str.contains(
            date_pattern, case=False, na=False, regex=True).astype(np.int8)
        pbar.update(1)

        term_pattern = r"terminat|cancel|expir|end\s+of\s+term"
        df["has_term_language"] = df["context"].str.contains(
            term_pattern, case=False, na=False, regex=True).astype(np.int8)
        pbar.update(1)

        def answer_overlap(row):
            if not row["answer"] or not row["context"]:
                return 0.0
            ans_words = set(row["answer"].lower().split())
            ctx_words = set(row["context"].lower().split())
            return len(ans_words & ctx_words) / (len(ans_words) + 1e-5)

        df["answer_context_overlap"] = df[["answer","context"]].apply(
            answer_overlap, axis=1).astype(np.float32)
        pbar.update(1)

        df["context_sent_count"] = df["context"].str.count(r"[.!?]+").astype(np.int32)
        pbar.update(1)

        df["log_context_len"] = np.log1p(df["context_len"]).astype(np.float32)
        df["log_answer_len"]  = np.log1p(df["answer_len"]).astype(np.float32)
        pbar.update(1)

        numeric_feat = [
            "context_len","answer_len","num_answers","context_density",
            "context_word_count","context_sent_count","log_context_len","log_answer_len"
        ]
        scaler = RobustScaler()
        X_num = scaler.fit_transform(df[numeric_feat].fillna(0))
        pca = PCA(n_components=2, random_state=CFG["seed"])
        pca_coords = pca.fit_transform(X_num)
        df["pca_1"] = pca_coords[:, 0].astype(np.float32)
        df["pca_2"] = pca_coords[:, 1].astype(np.float32)
        pbar.update(1)

    fig, ax = plt.subplots(figsize=(10, 7))
    colors = {"HIGH":"#d32f2f","MEDIUM":"#f57c00","LOW":"#388e3c"}
    for lvl, grp in df.groupby("risk_level"):
        ax.scatter(grp["pca_1"], grp["pca_2"], label=lvl,
                   color=colors.get(lvl,"gray"), alpha=0.4, s=15)
    ax.set_title("PCA Projection of Contract QA Features by Risk Level", fontsize=13)
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
    ax.legend()
    plt.tight_layout()
    save_plot(fig, "05_pca_risk_features")

    force_cleanup(X_num, pca_coords)
    logger.info(f"   Feature engineering complete. Total features: {len(df.columns)}")
    log_memory("after_features")
    return df, scaler, pca

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 8: FOUR-WAY DATA SPLIT (STRICT HOLDOUT)
# ─────────────────────────────────────────────────────────────────────────────
def verify_data_splits(splits: Dict[str, pd.DataFrame]) -> bool:
    names = list(splits.keys())
    for i in range(len(names)):
        for j in range(i+1, len(names)):
            a, b = splits[names[i]], splits[names[j]]
            overlap = set(a["qa_id"]) & set(b["qa_id"])
            if overlap:
                logger.error(f"  ✗ DATA LEAK: {names[i]} ∩ {names[j]} = {len(overlap)} samples!")
                return False
    logger.info("   No data leakage detected across all splits")
    return True


def create_four_way_split(df: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    logger.info("=" * 60)
    logger.info("STEP 5: FOUR-WAY DATA SPLIT (40/15/15/30)")
    logger.info("=" * 60)

    set_seed(CFG["seed"])
    label_col = "has_answer"

    df_main, df_holdout = train_test_split(
        df, test_size=CFG["holdout_ratio"],
        stratify=df[label_col], random_state=CFG["seed"]
    )

    test_frac = CFG["test_ratio"] / (1 - CFG["holdout_ratio"])
    df_main, df_test = train_test_split(
        df_main, test_size=test_frac,
        stratify=df_main[label_col], random_state=CFG["seed"]
    )

    val_frac = CFG["val_ratio"] / (1 - CFG["holdout_ratio"] - CFG["test_ratio"])
    df_train, df_val = train_test_split(
        df_main, test_size=val_frac,
        stratify=df_main[label_col], random_state=CFG["seed"]
    )

    splits = {"train": df_train, "val": df_val, "test": df_test, "holdout": df_holdout}

    total = len(df)
    for name, split in splits.items():
        pct = len(split) / total * 100
        pos = split[label_col].mean() * 100
        logger.info(f"  {name:8s}: {len(split):6,} rows ({pct:5.1f}%) | pos_rate={pos:.1f}%")

    assert verify_data_splits(splits), "DATA SPLIT INTEGRITY FAILED"
    logger.info(f"   Four-way split complete. Total rows = {total:,}")

    fig, ax = plt.subplots(figsize=(8, 5))
    labels = [f"{n}\n({len(s):,})" for n, s in splits.items()]
    sizes  = [len(s) for s in splits.values()]
    colors = ["#5c6bc0","#26a69a","#ef5350","#ffa726"]
    wedges, texts, autotexts = ax.pie(
        sizes, labels=labels, autopct="%1.1f%%", colors=colors,
        startangle=90, pctdistance=0.8
    )
    for t in autotexts:
        t.set_fontsize(10)
    ax.set_title("Four-Way Dataset Split", fontsize=14, fontweight="bold")
    plt.tight_layout()
    save_plot(fig, "06_data_splits")

    gc.collect()
    return splits

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 9: CLASSICAL ML BASELINE (ENSEMBLE)
# ─────────────────────────────────────────────────────────────────────────────
FEATURE_COLS = [
    "context_len","answer_len","num_answers","is_impossible",
    "context_density","context_word_count","context_sent_count",
    "log_context_len","log_answer_len","answer_context_overlap",
    "is_risk_high","is_risk_medium","is_risk_low",
    "has_money_terms","has_date_terms","has_term_language","q_is_extraction",
    "pca_1","pca_2","category_label","risk_label"
]

def train_classical_ensemble(splits: Dict[str, pd.DataFrame]) -> Dict:
    logger.info("=" * 60)
    logger.info("STEP 6: CLASSICAL ML ENSEMBLE TRAINING")
    logger.info("=" * 60)

    target = "has_answer"
    X_train = splits["train"][FEATURE_COLS].values
    y_train = splits["train"][target].values
    X_val   = splits["val"][FEATURE_COLS].values
    y_val   = splits["val"][target].values
    X_test  = splits["test"][FEATURE_COLS].values
    y_test  = splits["test"][target].values
    X_hold  = splits["holdout"][FEATURE_COLS].values
    y_hold  = splits["holdout"][target].values

    cw = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
    class_weight_dict = {i: cw[i] for i in range(len(cw))}
    logger.info(f"  Class weights: {class_weight_dict}")

    scaler = RobustScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_val_s   = scaler.transform(X_val)
    X_test_s  = scaler.transform(X_test)
    X_hold_s  = scaler.transform(X_hold)

    logger.info("  Hyperparameter tuning: RandomForest ...")
    rf_param_grid = {
        "n_estimators"    : [100, 200, 300],
        "max_depth"       : [None, 10, 20],
        "min_samples_split": [2, 5, 10],
        "max_features"    : ["sqrt", "log2"],
    }
    rf_base = RandomForestClassifier(class_weight="balanced",
                                     random_state=CFG["seed"], n_jobs=-1)
    rf_search = RandomizedSearchCV(
        rf_base, rf_param_grid, n_iter=8, cv=3, scoring="f1",
        random_state=CFG["seed"], n_jobs=-1, verbose=0
    )
    with tqdm(total=1, desc="  RF RandomSearch"):
        rf_search.fit(X_train_s, y_train)
    best_rf = rf_search.best_estimator_
    logger.info(f"  Best RF params: {rf_search.best_params_}")

    logger.info("  Hyperparameter tuning: GradientBoosting ...")
    gbm_param_grid = {
        "n_estimators"   : [100, 200],
        "learning_rate"  : [0.05, 0.1, 0.2],
        "max_depth"      : [3, 5, 7],
        "subsample"      : [0.7, 0.9, 1.0],
    }
    gbm_base = GradientBoostingClassifier(random_state=CFG["seed"])
    gbm_search = RandomizedSearchCV(
        gbm_base, gbm_param_grid, n_iter=8, cv=3, scoring="f1",
        random_state=CFG["seed"], n_jobs=-1, verbose=0
    )
    with tqdm(total=1, desc="  GBM RandomSearch"):
        gbm_search.fit(X_train_s, y_train)
    best_gbm = gbm_search.best_estimator_
    logger.info(f"  Best GBM params: {gbm_search.best_params_}")

    lr_clf = LogisticRegression(max_iter=500, class_weight="balanced",
                                random_state=CFG["seed"], C=1.0)
    with tqdm(total=1, desc="  LR Training"):
        lr_clf.fit(X_train_s, y_train)

    ensemble = VotingClassifier(
        estimators=[("rf", best_rf), ("gbm", best_gbm), ("lr", lr_clf)],
        voting="soft", n_jobs=-1
    )
    with tqdm(total=1, desc="  Ensemble Fitting"):
        ensemble.fit(X_train_s, y_train)

    results = {}
    datasets = {
        "train"  : (X_train_s, y_train),
        "val"    : (X_val_s,   y_val),
        "test"   : (X_test_s,  y_test),
        "holdout": (X_hold_s,  y_hold),
    }

    logger.info("\n  ── Ensemble Evaluation ──────────────────────────")
    with tqdm(datasets.items(), desc="  Evaluating splits") as splits_iter:
        for split_name, (X, y) in splits_iter:
            preds = ensemble.predict(X)
            proba = ensemble.predict_proba(X)[:, 1]
            acc   = accuracy_score(y, preds)
            f1    = f1_score(y, preds, average="weighted", zero_division=0)
            prec  = precision_score(y, preds, average="weighted", zero_division=0)
            rec   = recall_score(y, preds, average="weighted", zero_division=0)
            try:
                auc = roc_auc_score(y, proba)
            except Exception:
                auc = 0.0
            results[split_name] = {
                "accuracy": acc, "f1": f1, "precision": prec,
                "recall": rec, "auc": auc, "preds": preds, "proba": proba, "labels": y
            }
            logger.info(f"  {split_name:8s} | acc={acc:.4f} | f1={f1:.4f} | auc={auc:.4f}")

    train_acc = results["train"]["accuracy"]
    val_acc   = results["val"]["accuracy"]
    gap = train_acc - val_acc
    if gap > 0.10:
        logger.warning(f"  ⚠ OVERFITTING DETECTED: train-val gap = {gap:.4f}")
    else:
        logger.info(f"   Generalisation gap OK: {gap:.4f}")

    best_path = f"{CFG['models_dir']}/ensemble_classifier_{TIMESTAMP}.pkl"
    import pickle
    with open(best_path, "wb") as f:
        pickle.dump({"model": ensemble, "scaler": scaler,
                     "features": FEATURE_COLS, "results": results}, f)
    logger.info(f"   Ensemble saved → {best_path}")

    force_cleanup(X_train_s, X_val_s, X_test_s, X_hold_s)
    return ensemble, scaler, results

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 10: EVALUATION PLOTS (ALL SPLITS)
# ─────────────────────────────────────────────────────────────────────────────
def plot_all_evaluation(results: Dict, model_name: str = "Ensemble"):
    logger.info("  Generating evaluation plots …")

    metrics = ["accuracy","f1","precision","recall","auc"]
    split_names = list(results.keys())
    metric_vals = {m: [results[s][m] for s in split_names] for m in metrics}

    x = np.arange(len(split_names))
    width = 0.15
    fig, ax = plt.subplots(figsize=(14, 6))
    colors = ["#5c6bc0","#26a69a","#ef5350","#ffa726","#ab47bc"]
    for i, (met, col) in enumerate(zip(metrics, colors)):
        vals = metric_vals[met]
        bars = ax.bar(x + i*width, vals, width, label=met.capitalize(), color=col, alpha=0.85)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, v + 0.005,
                    f"{v:.3f}", ha="center", va="bottom", fontsize=7, rotation=90)

    ax.set_title(f"{model_name} — Performance Across All Splits", fontsize=13, fontweight="bold")
    ax.set_xticks(x + width*2)
    ax.set_xticklabels([s.capitalize() for s in split_names], fontsize=11)
    ax.set_ylabel("Score")
    ax.set_ylim(0, 1.15)
    ax.legend(loc="lower right")
    ax.axhline(0.99, color="red", linestyle="--", alpha=0.5, label="Target 0.99")
    plt.tight_layout()
    save_plot(fig, f"07_{model_name.lower()}_split_metrics")

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for idx, split_name in enumerate(["test", "holdout"]):
        if split_name not in results:
            continue
        cm = confusion_matrix(results[split_name]["labels"], results[split_name]["preds"])
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[idx],
                    xticklabels=["No Ans","Has Ans"],
                    yticklabels=["No Ans","Has Ans"])
        acc = results[split_name]["accuracy"]
        axes[idx].set_title(f"{split_name.capitalize()} Confusion Matrix\n(acc={acc:.4f})",
                            fontsize=12)
        axes[idx].set_ylabel("True")
        axes[idx].set_xlabel("Predicted")
    plt.suptitle("Confusion Matrices — Test & Holdout (Unseen) Data",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    save_plot(fig, "08_confusion_matrices_test_holdout")

    fig, ax = plt.subplots(figsize=(8, 5))
    for met, col in zip(["accuracy","f1","auc"], ["#5c6bc0","#26a69a","#ef5350"]):
        if "train" in results and "holdout" in results:
            gap = results["train"][met] - results["holdout"][met]
            ax.bar(met, gap, color=col, alpha=0.8, label=f"gap={gap:.4f}")
    ax.axhline(0.10, color="red", linestyle="--", label="Overfit Threshold (0.10)")
    ax.set_title("Generalisation Gap: Train vs Holdout", fontsize=13, fontweight="bold")
    ax.set_ylabel("Score Difference")
    ax.legend()
    plt.tight_layout()
    save_plot(fig, "09_generalization_gap")

    logger.info("   Evaluation plots saved")

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 11: ATTENTION-BASED NEURAL NETWORK (PyTorch)
# ─────────────────────────────────────────────────────────────────────────────
class ContractAttentionNet(nn.Module):
    def __init__(self, input_dim: int, n_classes: int = 2,
                 dropout: float = 0.35, noise_std: float = 0.05):
        super().__init__()
        self.noise_std = noise_std

        self.input_bn  = nn.BatchNorm1d(input_dim)
        self.proj      = nn.Linear(input_dim, 128)
        self.proj_bn   = nn.BatchNorm1d(128)
        self.proj_drop = nn.Dropout(dropout)

        self.attn_q  = nn.Linear(128, 64)
        self.attn_k  = nn.Linear(128, 64)
        self.attn_v  = nn.Linear(128, 128)
        self.attn_bn = nn.BatchNorm1d(128)

        self.fc1 = nn.Linear(128, 64)
        self.bn1 = nn.BatchNorm1d(64)
        self.d1  = nn.Dropout(dropout)

        self.fc2 = nn.Linear(64, 32)
        self.bn2 = nn.BatchNorm1d(32)
        self.d2  = nn.Dropout(dropout * 0.8)

        self.skip = nn.Linear(128, 32)
        self.out = nn.Linear(32, n_classes)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        if self.training:
            noise = torch.randn(x.shape, dtype=x.dtype, device=torch.device("cpu"))
            x = x + noise.to(x.device) * self.noise_std

        x = self.input_bn(x)
        h = F.gelu(self.proj_bn(self.proj(x)))
        h = self.proj_drop(h)

        q = self.attn_q(h)
        k = self.attn_k(h)
        v = self.attn_v(h)
        score = torch.sum(q * k, dim=-1, keepdim=True) / (64 ** 0.5)
        attn  = torch.sigmoid(score)
        h_attn = attn * v
        h = self.attn_bn(h + h_attn)

        res  = self.skip(h)
        h    = F.gelu(self.bn1(self.fc1(h)))
        h    = self.d1(h)
        h    = F.gelu(self.bn2(self.fc2(h)))
        h    = self.d2(h)
        h    = h + res
        return self.out(h)


class ContractDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


def mixup_data(x, y, alpha=0.2):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=torch.device("cpu")).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[idx]
    y_a, y_b = y, y[idx]
    return mixed_x, y_a, y_b, lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


def _probe_cuda_device() -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")
    try:
        a = torch.tensor([1.0], device="cuda")
        b = a + 1
        del a, b
        torch.cuda.empty_cache()
        return torch.device("cuda")
    except Exception as e:
        logger.warning(
            f"  ⚠ CUDA probe failed ({type(e).__name__}: {e}). "
            "Falling back to CPU — all NN ops will run on CPU."
        )
        torch.cuda.empty_cache()
        return torch.device("cpu")


def train_attention_nn(splits: Dict[str, pd.DataFrame],
                       scaler_ref=None) -> Dict:
    logger.info("=" * 60)
    logger.info("STEP 7: ATTENTION NEURAL NETWORK TRAINING")
    logger.info("=" * 60)

    target = "has_answer"
    device = _probe_cuda_device()
    logger.info(f"  Device: {device}")

    scaler_nn = RobustScaler()
    X_train = scaler_nn.fit_transform(splits["train"][FEATURE_COLS].fillna(0).values)
    y_train = splits["train"][target].values

    X_val = scaler_nn.transform(splits["val"][FEATURE_COLS].fillna(0).values)
    y_val = splits["val"][target].values
    X_test = scaler_nn.transform(splits["test"][FEATURE_COLS].fillna(0).values)
    y_test = splits["test"][target].values
    X_hold = scaler_nn.transform(splits["holdout"][FEATURE_COLS].fillna(0).values)
    y_hold = splits["holdout"][target].values

    cw = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
    weights_tensor = torch.tensor(cw, dtype=torch.float32).to(device)

    sample_weights = torch.tensor([cw[y] for y in y_train], dtype=torch.float32)
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(y_train), replacement=True)

    train_ds = ContractDataset(X_train, y_train)
    val_ds   = ContractDataset(X_val,   y_val)

    train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"],
                              sampler=sampler, drop_last=True)
    val_loader   = DataLoader(val_ds, batch_size=CFG["batch_size"]*4, shuffle=False)

    input_dim = X_train.shape[1]
    model = ContractAttentionNet(input_dim=input_dim, n_classes=2,
                                 dropout=CFG["dropout"]).to(device)
    logger.info(f"  Model parameters: {sum(p.numel() for p in model.parameters()):,}")

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"]
    )
    criterion = nn.CrossEntropyLoss(weight=weights_tensor, label_smoothing=0.05)
    scheduler = ReduceLROnPlateau(optimizer, mode="max", factor=0.5,
                                  patience=2, min_lr=1e-6)

    best_val_acc = 0.0
    best_state   = None
    patience_cnt = 0
    train_accs, val_accs, train_losses, val_losses = [], [], [], []

    with tqdm(range(CFG["epochs"]), desc="  Training epochs") as epoch_bar:
        for epoch in epoch_bar:
            model.train()
            ep_loss, ep_correct, ep_total = 0.0, 0, 0
            for X_b, y_b in train_loader:
                X_b, y_b = X_b.to(device), y_b.to(device)
                X_mix, y_a, y_b_mix, lam = mixup_data(X_b, y_b, alpha=0.2)
                optimizer.zero_grad()
                logits = model(X_mix)
                loss   = mixup_criterion(criterion, logits, y_a, y_b_mix, lam)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

                ep_loss    += loss.item() * X_b.size(0)
                preds       = logits.argmax(dim=1)
                ep_correct += (preds == y_a).sum().item()
                ep_total   += X_b.size(0)

            train_loss = ep_loss / ep_total
            train_acc  = ep_correct / ep_total

            model.eval()
            v_loss, v_correct, v_total = 0.0, 0, 0
            with torch.no_grad():
                for X_b, y_b in val_loader:
                    X_b, y_b = X_b.to(device), y_b.to(device)
                    logits   = model(X_b)
                    loss_v   = criterion(logits, y_b)
                    v_loss  += loss_v.item() * X_b.size(0)
                    preds    = logits.argmax(dim=1)
                    v_correct += (preds == y_b).sum().item()
                    v_total   += X_b.size(0)

            val_loss = v_loss / v_total
            val_acc  = v_correct / v_total

            train_accs.append(train_acc)
            val_accs.append(val_acc)
            train_losses.append(train_loss)
            val_losses.append(val_loss)

            scheduler.step(val_acc)

            epoch_bar.set_postfix({
                "tr_loss": f"{train_loss:.4f}", "tr_acc": f"{train_acc:.4f}",
                "vl_acc": f"{val_acc:.4f}"
            })

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_state   = copy.deepcopy(model.state_dict())
                patience_cnt = 0
                logger.debug(f"     New best val_acc={best_val_acc:.4f} at epoch {epoch+1}")
            else:
                patience_cnt += 1

            if patience_cnt >= CFG["early_stop_patience"]:
                logger.info(f"  Early stopping triggered at epoch {epoch+1}")
                break

    model.load_state_dict(best_state)
    torch.save(best_state, f"{CFG['models_dir']}/attention_nn_best_{TIMESTAMP}.pt")
    logger.info(f"   Best NN model saved (val_acc={best_val_acc:.4f})")

    nn_results = {}
    all_data = {
        "train"  : (X_train, y_train),
        "val"    : (X_val,   y_val),
        "test"   : (X_test,  y_test),
        "holdout": (X_hold,  y_hold),
    }
    model.eval()
    logger.info("\n  ── NN Evaluation ────────────────────────────────")
    with tqdm(all_data.items(), desc="  Evaluating NN") as split_it:
        for split_name, (X, y) in split_it:
            ds  = ContractDataset(X, y)
            dl  = DataLoader(ds, batch_size=CFG["batch_size"]*4, shuffle=False)
            all_preds, all_proba, all_labels = [], [], []
            with torch.no_grad():
                for X_b, y_b in dl:
                    X_b = X_b.to(device)
                    logits = model(X_b)
                    proba  = F.softmax(logits, dim=1)[:, 1].cpu().numpy()
                    preds  = logits.argmax(dim=1).cpu().numpy()
                    all_preds.extend(preds)
                    all_proba.extend(proba)
                    all_labels.extend(y_b.numpy())

            preds  = np.array(all_preds)
            proba  = np.array(all_proba)
            labels = np.array(all_labels)
            acc  = accuracy_score(labels, preds)
            f1   = f1_score(labels, preds, average="weighted", zero_division=0)
            prec = precision_score(labels, preds, average="weighted", zero_division=0)
            rec  = recall_score(labels, preds, average="weighted", zero_division=0)
            try:
                auc = roc_auc_score(labels, proba)
            except Exception:
                auc = 0.0
            nn_results[split_name] = {
                "accuracy": acc, "f1": f1, "precision": prec,
                "recall": rec, "auc": auc, "preds": preds, "proba": proba, "labels": labels
            }
            logger.info(f"  {split_name:8s} | acc={acc:.4f} | f1={f1:.4f} | auc={auc:.4f}")

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ep = range(1, len(train_accs)+1)
    axes[0].plot(ep, train_accs, "b-o", ms=4, label="Train Acc")
    axes[0].plot(ep, val_accs,   "r-o", ms=4, label="Val Acc")
    best_ep = int(np.argmax(val_accs)) + 1
    axes[0].axvline(best_ep, color="green", linestyle="--",
                    label=f"Best @ ep{best_ep} ({max(val_accs):.4f})")
    axes[0].set_title("Accuracy Curves", fontsize=12)
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Accuracy")
    axes[0].legend()

    axes[1].plot(ep, train_losses, "b-o", ms=4, label="Train Loss")
    axes[1].plot(ep, val_losses,   "r-o", ms=4, label="Val Loss")
    axes[1].set_title("Loss Curves", fontsize=12)
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Loss")
    axes[1].legend()

    plt.suptitle("Attention NN — Training Dynamics", fontsize=13, fontweight="bold")
    plt.tight_layout()
    save_plot(fig, "10_nn_training_curves")

    force_cleanup(X_train, X_val, X_test, X_hold, train_ds, val_ds)
    log_memory("after_nn_train")
    return model, scaler_nn, nn_results

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 12: LLM FINE-TUNING WITH LORA (PEFT) – RESOURCE‑AWARE SKIP
# ─────────────────────────────────────────────────────────────────────────────
def build_lora_finetune_dataset(df: pd.DataFrame) -> HFDataset:
    instruction_template = (
        "You are a legal contract analysis assistant. "
        "Review the following contract excerpt and answer the question.\n\n"
        "CONTRACT EXCERPT:\n{context}\n\n"
        "QUESTION: {question}\n\n"
        "ANSWER:"
    )
    records = []
    sample = df[df["has_answer"] == 1].sample(
        min(500, len(df[df["has_answer"] == 1])),
        random_state=CFG["seed"]
    )
    for _, row in tqdm(sample.iterrows(), total=len(sample),
                        desc="  Building fine-tune dataset"):
        prompt = instruction_template.format(
            context=row["context"][:800],
            question=row["question"]
        )
        response = (row["answer"][:300] if row["answer"]
                    else "This clause is not present in the contract.")
        records.append({
            "prompt"  : prompt,
            "response": response,
            "text"    : f"{prompt} {response}",
            "category": row["category"],
            "risk"    : row["risk_level"],
        })
    return HFDataset.from_list(records)


def load_model_with_quantization(model_name: str, token_path: str):
    device = _probe_cuda_device()
    if device.type == "cpu":
        logger.warning("  ⚠ No usable CUDA device – skipping LLM fine-tuning (would be too slow on CPU)")
        return None, None

    total, avail = get_system_memory()
    if avail < 14.0:
        logger.warning(f"  ⚠ Insufficient RAM ({avail:.1f}GB available) for 7B LLM. Need ~14GB free. Skipping.")
        return None, None

    logger.info(f"  Loading model: {model_name}")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    try:
        tokenizer = AutoTokenizer.from_pretrained(
            model_name, trust_remote_code=True,
            padding_side="right"
        )
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True,
            torch_dtype=torch.float16,
        )
        model.config.use_cache = False
        model.config.pretraining_tp = 1
        logger.info(f"   Model loaded: {model_name}")
        return tokenizer, model
    except Exception as e:
        logger.error(f"  ✗ Failed to load {model_name}: {e}")
        return None, None


def apply_lora(model, r: int = 8, alpha: int = 16, dropout: float = 0.05):
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=r,
        lora_alpha=alpha,
        lora_dropout=dropout,
        bias="none",
        target_modules=["q_proj","k_proj","v_proj","o_proj",
                         "gate_proj","up_proj","down_proj"],
    )
    model = get_peft_model(model, lora_config)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    logger.info(f"  LoRA: trainable={trainable:,} / total={total:,} ({100*trainable/total:.2f}%)")
    return model


class LLMContractDataset(Dataset):
    def __init__(self, hf_dataset: HFDataset, tokenizer, max_length: int):
        self.data      = hf_dataset
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        enc  = self.tokenizer(
            item["text"],
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        input_ids      = enc["input_ids"].squeeze()
        attention_mask = enc["attention_mask"].squeeze()
        labels         = input_ids.clone()
        labels[attention_mask == 0] = -100
        return {
            "input_ids"     : input_ids,
            "attention_mask": attention_mask,
            "labels"        : labels,
        }


def fine_tune_llm(df: pd.DataFrame, model_name: str,
                  output_dir: str = CFG["finetuned_dir"]) -> bool:
    logger.info("=" * 60)
    logger.info(f"STEP 8: LLM FINE-TUNING — {model_name}")
    logger.info("=" * 60)
    log_memory("before_finetune")

    if not memory_safe():
        logger.warning("  ⚠ Low system memory – skipping LLM fine-tuning")
        return False

    device = _probe_cuda_device()
    if device.type != "cuda":
        logger.warning("  ⚠ No compatible GPU – skipping fine-tuning (would be too slow)")
        return False

    tokenizer, base_model = load_model_with_quantization(model_name, CFG["hf_token_path"])
    if base_model is None:
        logger.error("  ✗ Could not load model, skipping fine-tuning")
        return False

    model = apply_lora(base_model, r=CFG["lora_r"],
                       alpha=CFG["lora_alpha"], dropout=CFG["lora_dropout"])

    hf_ds = build_lora_finetune_dataset(df)
    if len(hf_ds) < 10:
        logger.warning("  Dataset too small for fine-tuning, skipping")
        return False

    n_train = int(len(hf_ds) * 0.85)
    train_ds = LLMContractDataset(hf_ds.select(range(n_train)), tokenizer, CFG["max_length"])
    eval_ds  = LLMContractDataset(hf_ds.select(range(n_train, len(hf_ds))), tokenizer, CFG["max_length"])

    # FIX: removed 'save_safetensors' argument (not in older transformers)
    training_args = TrainingArguments(
        output_dir                  = output_dir,
        num_train_epochs            = 2,
        per_device_train_batch_size = 1,
        per_device_eval_batch_size  = 1,
        gradient_accumulation_steps = CFG["grad_accum"],
        learning_rate               = CFG["lr"],
        weight_decay                = CFG["weight_decay"],
        warmup_ratio                = CFG["warmup_ratio"],
        lr_scheduler_type           = "cosine",
        eval_strategy               = "epoch",           # was 'evaluation_strategy'
        save_strategy               = "epoch",
        load_best_model_at_end      = True,
        metric_for_best_model       = "eval_loss",
        greater_is_better           = False,
        logging_steps               = 10,
        fp16                        = True,
        report_to                   = "none",
        dataloader_num_workers      = 0,
        remove_unused_columns       = False,
    )

    trainer = Trainer(
        model         = model,
        args          = training_args,
        train_dataset = train_ds,
        eval_dataset  = eval_ds,
        callbacks     = [EarlyStoppingCallback(early_stopping_patience=1)],
    )

    logger.info("  Starting LoRA fine-tuning …")
    try:
        with tqdm(total=1, desc="  Fine-tuning LLM"):
            trainer.train()
        model.save_pretrained(output_dir)
        tokenizer.save_pretrained(output_dir)
        base_model.config.save_pretrained(output_dir)
        logger.info(f"   Fine-tuned model saved → {output_dir}")
    except Exception as e:
        logger.error(f"  ✗ Fine-tuning error: {traceback.format_exc()}")
        return False
    finally:
        force_cleanup(model, base_model, tokenizer, train_ds, eval_ds)

    log_memory("after_finetune")
    return True

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 13: CONTRACT REVIEW AI AGENT (unchanged)
# ─────────────────────────────────────────────────────────────────────────────
GOLD_STANDARD_PLAYBOOK = {
    "Liability Cap": {
        "standard"   : "Mutual cap at 12 months of fees paid",
        "risk_flags" : ["unlimited", "no cap", "uncapped"],
        "preferred"  : "limited to fees paid in preceding 12 months",
    },
    "Termination For Convenience": {
        "standard"   : "30-day written notice required",
        "risk_flags" : ["immediate", "without notice", "at will"],
        "preferred"  : "30 days written notice",
    },
    "Governing Law": {
        "standard"   : "Neutral jurisdiction (Delaware or New York)",
        "risk_flags" : ["counterparty state only", "foreign jurisdiction"],
        "preferred"  : "State of Delaware or New York",
    },
    "Renewal Term": {
        "standard"   : "Written consent required for auto-renewal",
        "risk_flags" : ["automatic renewal", "auto-renews", "unless terminated"],
        "preferred"  : "written opt-in for renewal",
    },
    "IP Ownership Assignment": {
        "standard"   : "Client retains all IP for work-for-hire",
        "risk_flags" : ["assigns all IP", "vendor retains", "background IP"],
        "preferred"  : "All work product is work-for-hire and owned by client",
    },
}


class ContractClauseExtractor:
    CLAUSE_PATTERNS = {
        "Liability Cap"       : r"liabilit[y|ies].{0,200}(cap|limit|not exceed|maximum).{0,100}",
        "Termination"         : r"terminat.{0,300}(notice|days|written).{0,100}",
        "Governing Law"       : r"govern.{0,50}(law|jurisdiction).{0,200}",
        "Renewal Term"        : r"renew.{0,200}(term|period|automatic|auto).{0,100}",
        "IP Ownership"        : r"intellectual property.{0,300}(own|assign|vest|retain).{0,100}",
        "Confidentiality"     : r"confidential.{0,300}(obligation|duty|term|period).{0,100}",
        "Indemnification"     : r"indemn.{0,400}(loss|damage|claim|cost).{0,100}",
        "Warranty"            : r"warrant.{0,300}(period|duration|year|month).{0,100}",
        "Dispute Resolution"  : r"disput.{0,200}(arbitrat|mediat|litigation|court).{0,100}",
        "Force Majeure"       : r"force majeure.{0,300}",
    }

    def extract_clauses(self, contract_text: str) -> Dict[str, List[str]]:
        import re
        clauses = {}
        for clause_name, pattern in tqdm(self.CLAUSE_PATTERNS.items(),
                                          desc="  Extracting clauses", leave=False):
            matches = re.findall(pattern, contract_text, re.IGNORECASE | re.DOTALL)
            clauses[clause_name] = [m.strip()[:300] for m in matches[:3]]
        return clauses


class ContractRedliner:
    def __init__(self):
        self.extractor = ContractClauseExtractor()

    def compare_against_playbook(self, clauses: Dict[str, List[str]]) -> Dict[str, Dict]:
        results = {}
        for clause_name, playbook_entry in GOLD_STANDARD_PLAYBOOK.items():
            found_texts = clauses.get(clause_name, clauses.get(clause_name.split()[0], []))
            risk_flags_found = []
            for flag in playbook_entry["risk_flags"]:
                for text in found_texts:
                    if flag.lower() in text.lower():
                        risk_flags_found.append(flag)
            is_nonstandard = len(risk_flags_found) > 0 or len(found_texts) == 0
            risk = "HIGH" if clause_name in RISK_LEVELS["HIGH"] else (
                   "MEDIUM" if clause_name in RISK_LEVELS["MEDIUM"] else "LOW")
            results[clause_name] = {
                "found"          : len(found_texts) > 0,
                "extracted_text" : found_texts,
                "is_nonstandard" : is_nonstandard,
                "risk_level"     : risk if is_nonstandard else "OK",
                "risk_flags"     : risk_flags_found,
                "standard"       : playbook_entry["standard"],
                "preferred"      : playbook_entry["preferred"],
                "redline_needed" : is_nonstandard,
                "plain_english"  : self._explain_risk(clause_name, risk_flags_found,
                                                        found_texts, playbook_entry),
            }
        return results

    def _explain_risk(self, clause: str, flags: List[str],
                      texts: List[str], playbook: Dict) -> str:
        if not texts:
            return (f"⚠ {clause} clause is MISSING. Standard practice requires: "
                    f"{playbook['standard']}. Recommend adding preferred language: "
                    f"'{playbook['preferred']}'.")
        if flags:
            return (f" {clause} contains non-standard language ({', '.join(flags)}). "
                    f"This deviates from standard: '{playbook['standard']}'. "
                    f"Recommended language: '{playbook['preferred']}'.")
        return f" {clause} appears standard. Review for specific terms."

    def generate_redlines(self, comparison: Dict[str, Dict]) -> List[Dict]:
        redlines = []
        for clause, details in comparison.items():
            if details["redline_needed"]:
                redlines.append({
                    "clause"       : clause,
                    "risk_level"   : details["risk_level"],
                    "issue"        : f"Non-standard {clause}",
                    "current_text" : (details["extracted_text"][0][:200]
                                      if details["extracted_text"] else "MISSING"),
                    "recommended"  : details["preferred"],
                    "explanation"  : details["plain_english"],
                })
        order = {"HIGH": 0, "MEDIUM": 1, "LOW": 2, "OK": 3}
        redlines.sort(key=lambda x: order.get(x["risk_level"], 9))
        return redlines

    def generate_negotiation_checklist(self, redlines: List[Dict]) -> str:
        lines = ["=" * 60,
                 "NEGOTIATION CHECKLIST",
                 "=" * 60,
                 f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}",
                 f"Total issues: {len(redlines)}",
                 "",
                 "PRIORITY ACTION ITEMS:",
                 ""]
        for i, item in enumerate(redlines, 1):
            lines.append(f"{i}. [{item['risk_level']}] {item['clause']}")
            lines.append(f"   Issue: {item['issue']}")
            lines.append(f"   Recommend: {item['recommended']}")
            lines.append("")
        return "\n".join(lines)

    def generate_email_summary(self, contract_name: str,
                                redlines: List[Dict]) -> str:
        high   = [r for r in redlines if r["risk_level"] == "HIGH"]
        medium = [r for r in redlines if r["risk_level"] == "MEDIUM"]
        email = f"""Subject: Contract Review — {contract_name} [ACTION REQUIRED]

Dear [Counterparty],

Following our legal team's review of {contract_name}, we have identified
{len(redlines)} items requiring attention prior to execution.

CRITICAL ISSUES ({len(high)} items):
"""
        for item in high:
            email += f"  • {item['clause']}: {item['explanation'][:120]}\n"
        email += f"\nIMPORTANT ISSUES ({len(medium)} items):\n"
        for item in medium:
            email += f"  • {item['clause']}: {item['explanation'][:100]}\n"
        email += """
We propose revising the above sections in line with our standard playbook.
A redlined version is attached for your review.

Please confirm your availability for a call to discuss these points.

Best regards,
[Legal Counsel / Contract Management Team]
"""
        return email


class ContractReviewAgent:
    def __init__(self, llm_model=None, llm_tokenizer=None,
                 classifier=None, scaler=None):
        self.redliner   = ContractRedliner()
        self.llm_model  = llm_model
        self.llm_tok    = llm_tokenizer
        self.classifier = classifier
        self.scaler     = scaler
        logger.info("   ContractReviewAgent initialised")

    def llm_analyze(self, contract_text: str, question: str,
                    max_new_tokens: int = 256) -> str:
        if self.llm_model is None or self.llm_tok is None:
            return "[LLM not loaded — returning pattern-based analysis]"
        prompt = (
            "You are an expert legal contract reviewer.\n"
            f"CONTRACT:\n{contract_text[:1000]}\n\n"
            f"QUESTION: {question}\n\nANSWER:"
        )
        inputs = self.llm_tok(prompt, return_tensors="pt",
                               max_length=512, truncation=True).to(
            next(self.llm_model.parameters()).device
        )
        with torch.no_grad():
            outputs = self.llm_model.generate(
                **inputs, max_new_tokens=max_new_tokens,
                temperature=0.3, do_sample=True, top_p=0.9,
            )
        decoded = self.llm_tok.decode(outputs[0], skip_special_tokens=True)
        answer  = decoded[len(prompt):].strip()
        return answer

    def review_contract(self, contract_text: str,
                        contract_name: str = "Contract") -> Dict:
        logger.info(f"\n{'='*60}")
        logger.info(f"  REVIEWING: {contract_name}")
        logger.info(f"{'='*60}")

        clauses = self.redliner.extractor.extract_clauses(contract_text)
        logger.info(f"  Extracted {sum(len(v) for v in clauses.values())} clause snippets")

        llm_insights = {}
        for clause in ["Liability Cap", "Termination For Convenience", "IP Ownership"]:
            q = f"What does this contract say about {clause}? Is it standard or risky?"
            llm_insights[clause] = self.llm_analyze(contract_text, q)

        comparison = self.redliner.compare_against_playbook(clauses)
        redlines = self.redliner.generate_redlines(comparison)

        obligations = self.llm_analyze(
            contract_text,
            "List the top 5 obligations of each party under this contract."
        )
        dates = self.llm_analyze(
            contract_text,
            "What are the key dates: effective date, expiration, renewal deadlines?"
        )
        checklist = self.redliner.generate_negotiation_checklist(redlines)
        email = self.redliner.generate_email_summary(contract_name, redlines)

        result = {
            "contract_name"  : contract_name,
            "clauses"        : clauses,
            "llm_insights"   : llm_insights,
            "comparison"     : comparison,
            "redlines"       : redlines,
            "obligations"    : obligations,
            "key_dates"      : dates,
            "checklist"      : checklist,
            "email_summary"  : email,
            "risk_summary"   : {
                "total_issues" : len(redlines),
                "high_risk"    : sum(1 for r in redlines if r["risk_level"] == "HIGH"),
                "medium_risk"  : sum(1 for r in redlines if r["risk_level"] == "MEDIUM"),
                "low_risk"     : sum(1 for r in redlines if r["risk_level"] == "LOW"),
            }
        }
        self._print_review_report(result)
        return result

    def _print_review_report(self, result: Dict):
        logger.info("\n" + "─"*60)
        logger.info(f"  CONTRACT REVIEW REPORT: {result['contract_name']}")
        logger.info("─"*60)
        rs = result["risk_summary"]
        logger.info(f"  Issues: {rs['total_issues']} total | "
                    f"HIGH={rs['high_risk']} | MED={rs['medium_risk']} | LOW={rs['low_risk']}")
        logger.info("\n  REDLINES:")
        for r in result["redlines"][:5]:
            logger.info(f"    [{r['risk_level']}] {r['clause']}: {r['explanation'][:80]}…")
        logger.info("\n  CHECKLIST PREVIEW:")
        for line in result["checklist"].split("\n")[:10]:
            logger.info(f"    {line}")

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 14: FINAL EVALUATION REPORTS & PLOTS
# ─────────────────────────────────────────────────────────────────────────────
def plot_final_comparison(ensemble_results: Dict, nn_results: Dict):
    splits = ["train","val","test","holdout"]
    metrics = ["accuracy","f1","auc"]
    n_splits = len(splits)

    fig, axes = plt.subplots(1, len(metrics), figsize=(16, 5))
    fig.suptitle("Model Comparison: Ensemble vs Attention NN — All Splits",
                 fontsize=13, fontweight="bold")

    x = np.arange(n_splits)
    w = 0.35
    for mi, (met, ax) in enumerate(zip(metrics, axes)):
        ens_vals = [ensemble_results.get(s, {}).get(met, 0) for s in splits]
        nn_vals  = [nn_results.get(s, {}).get(met, 0) for s in splits]
        b1 = ax.bar(x - w/2, ens_vals, w, label="Ensemble", color="#5c6bc0", alpha=0.85)
        b2 = ax.bar(x + w/2, nn_vals,  w, label="Attn NN",  color="#ef5350", alpha=0.85)
        ax.set_title(met.capitalize(), fontsize=11)
        ax.set_xticks(x)
        ax.set_xticklabels([s.capitalize() for s in splits])
        ax.set_ylim(0, 1.1)
        ax.axhline(0.99, color="green", linestyle="--", alpha=0.5, linewidth=1)
        ax.legend(fontsize=8)
        for bar in list(b1) + list(b2):
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2, h + 0.005,
                    f"{h:.3f}", ha="center", va="bottom", fontsize=7)

    plt.tight_layout()
    save_plot(fig, "11_model_comparison_all_splits")

    fig, ax = plt.subplots(figsize=(9, 5))
    for mi, met in enumerate(metrics):
        ens_gap = (ensemble_results.get("train", {}).get(met, 0) -
                   ensemble_results.get("holdout", {}).get(met, 0))
        nn_gap  = (nn_results.get("train", {}).get(met, 0) -
                   nn_results.get("holdout", {}).get(met, 0))
        ax.bar(mi*3,   ens_gap, color="#5c6bc0", alpha=0.85, label="Ensemble" if mi == 0 else "")
        ax.bar(mi*3+1, nn_gap,  color="#ef5350", alpha=0.85, label="Attn NN"  if mi == 0 else "")

    ax.set_xticks([0.5, 3.5, 6.5])
    ax.set_xticklabels(metrics)
    ax.axhline(0.10, color="red", linestyle="--", label="Overfit Threshold")
    ax.axhline(0.0,  color="black", linewidth=0.5)
    ax.set_title("Generalisation Gap (Train–Holdout) by Model & Metric",
                 fontsize=12, fontweight="bold")
    ax.set_ylabel("Gap")
    ax.legend()
    plt.tight_layout()
    save_plot(fig, "12_generalization_gap_comparison")


def generate_text_report(ensemble_results: Dict, nn_results: Dict,
                          splits: Dict[str, pd.DataFrame]):
    lines = [
        "=" * 70,
        "CONTRACT REVIEW AI AGENT — PIPELINE REPORT",
        f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
        "=" * 70,
        "",
        "DATASET SUMMARY",
        "─" * 40,
    ]
    for name, df in splits.items():
        lines.append(f"  {name:10s}: {len(df):6,} samples | "
                     f"pos_rate={df['has_answer'].mean()*100:.1f}%")

    lines += ["", "ENSEMBLE MODEL PERFORMANCE", "─" * 40]
    for split, res in ensemble_results.items():
        lines.append(f"  {split:10s}: acc={res['accuracy']:.4f} | "
                     f"f1={res['f1']:.4f} | auc={res['auc']:.4f}")

    lines += ["", "ATTENTION NN PERFORMANCE", "─" * 40]
    for split, res in nn_results.items():
        lines.append(f"  {split:10s}: acc={res['accuracy']:.4f} | "
                     f"f1={res['f1']:.4f} | auc={res['auc']:.4f}")

    lines += ["", "OVERFITTING ANALYSIS", "─" * 40]
    for model_name, results in [("Ensemble", ensemble_results), ("Attn NN", nn_results)]:
        for met in ["accuracy","f1"]:
            gap = results.get("train",{}).get(met, 0) - results.get("holdout",{}).get(met, 0)
            status = "⚠ OVERFIT" if gap > 0.1 else " OK"
            lines.append(f"  {model_name} {met}: gap={gap:.4f} [{status}]")

    report_path = f"{CFG['output_dir']}/pipeline_report_{TIMESTAMP}.txt"
    with open(report_path, "w") as f:
        f.write("\n".join(lines))
    logger.info(f"   Report saved → {report_path}")
    return report_path

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 15: DEMO AGENT RUN
# ─────────────────────────────────────────────────────────────────────────────
SAMPLE_CONTRACT = """
NON-DISCLOSURE AGREEMENT

This Non-Disclosure Agreement ("Agreement") is entered into as of January 1, 2025,
between Acme Corp ("Disclosing Party") and Beta Inc ("Receiving Party").

1. CONFIDENTIALITY OBLIGATIONS
The Receiving Party agrees to keep all Confidential Information strictly
confidential for a period of five (5) years from the date of disclosure.

2. TERMINATION
Either party may terminate this Agreement immediately without notice and without
any cause whatsoever at its sole discretion.

3. GOVERNING LAW
This Agreement shall be governed by the laws of the Cayman Islands.

4. LIABILITY
In no event shall either party's liability exceed the greater of $100 or
the fees paid under this Agreement, with no cap on consequential damages.

5. IP OWNERSHIP
Any IP created during the engagement shall automatically vest in and be assigned
to the Vendor. The client waives all rights to work product.

6. RENEWAL
This Agreement automatically renews annually unless terminated, with no prior
notice required.
"""


def run_demo_agent(agent: ContractReviewAgent):
    logger.info("\n" + "=" * 60)
    logger.info("DEMO: Running contract review on sample NDA")
    logger.info("=" * 60)
    result = agent.review_contract(SAMPLE_CONTRACT, "Sample NDA 2025")

    out_path = f"{CFG['output_dir']}/demo_review_{TIMESTAMP}.json"
    with open(out_path, "w") as f:
        safe = {k: v for k, v in result.items()
                if k not in ["clauses", "comparison"]}
        json.dump(safe, f, indent=2, default=str)
    logger.info(f"   Demo review saved → {out_path}")

    with open(f"{CFG['output_dir']}/negotiation_checklist_{TIMESTAMP}.txt", "w") as f:
        f.write(result["checklist"])
    with open(f"{CFG['output_dir']}/email_summary_{TIMESTAMP}.txt", "w") as f:
        f.write(result["email_summary"])

    return result

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 16: MAIN ORCHESTRATOR
# ─────────────────────────────────────────────────────────────────────────────
def main():
    logger.info("\n" + "╔" + "═"*60 + "╗")
    logger.info("║   CONTRACT REVIEW & REDLINING AI AGENT PIPELINE         ║")
    logger.info("║   LegalTech SME Automation | CUAD + LLM Fine-Tuning     ║")
    logger.info("╚" + "═"*60 + "╝\n")

    set_seed(CFG["seed"])
    start_time = time.time()

    auth_ok = authenticate_huggingface()
    log_memory("start")

    master_df, raw_dfs = load_all_datasets()
    if master_df.empty:
        logger.error("No data loaded. Exiting.")
        return

    clean_df, le_risk, le_cat = preprocess_and_clean(master_df)
    force_cleanup(master_df, raw_dfs)

    run_eda(clean_df)
    gc.collect()

    feat_df, scaler, pca_model = engineer_features(clean_df)
    force_cleanup(clean_df)

    splits = create_four_way_split(feat_df)

    ensemble_model, ens_scaler, ens_results = train_classical_ensemble(splits)
    plot_all_evaluation(ens_results, "Ensemble")
    gc.collect()

    nn_model, nn_scaler, nn_results = train_attention_nn(splits)
    plot_all_evaluation(nn_results, "AttentionNN")
    gc.collect()

    plot_final_comparison(ens_results, nn_results)

    llm_ok = False
    llm_tokenizer = None
    llm_model_obj = None

    if auth_ok and memory_safe():
        for model_name in [CFG["primary_model"], CFG["backup_model"]]:
            logger.info(f"  Attempting fine-tuning with: {model_name}")
            llm_ok = fine_tune_llm(feat_df, model_name, CFG["finetuned_dir"])
            if llm_ok:
                break
            gc.collect()
    else:
        logger.warning("  Skipping LLM fine-tuning (auth failed or low memory)")

    ft_dir = Path(CFG["finetuned_dir"])
    if llm_ok and (ft_dir / "adapter_config.json").exists():
        try:
            logger.info("  Loading fine-tuned model for agent …")
            llm_tokenizer, base = load_model_with_quantization(
                CFG["primary_model"] if Path(CFG["primary_model"]).exists()
                else CFG["backup_model"],
                CFG["hf_token_path"]
            )
            if base is not None:
                llm_model_obj = PeftModel.from_pretrained(base, str(ft_dir))
                llm_model_obj.eval()
        except Exception as e:
            logger.warning(f"  Could not load fine-tuned model: {e}")

    agent = ContractReviewAgent(
        llm_model    = llm_model_obj,
        llm_tokenizer= llm_tokenizer,
        classifier   = ensemble_model,
        scaler       = ens_scaler,
    )

    demo_result = run_demo_agent(agent)
    generate_text_report(ens_results, nn_results, splits)

    elapsed = time.time() - start_time
    logger.info("\n" + "="*60)
    logger.info(f"   PIPELINE COMPLETE in {elapsed/60:.1f} minutes")
    logger.info(f"  Outputs → {CFG['output_dir']}")
    logger.info(f"  Plots   → {CFG['plots_dir']}")
    logger.info(f"  Models  → {CFG['models_dir']}")
    logger.info(f"  Fine-tuned model → {CFG['finetuned_dir']}")
    logger.info("="*60)

    force_cleanup(feat_df, nn_model, ensemble_model, llm_model_obj)
    return {
        "ensemble_results": ens_results,
        "nn_results"      : nn_results,
        "agent"           : agent,
        "demo"            : demo_result,
    }


if __name__ == "__main__":
    main()

2026-04-26 01:19:21,094 | INFO | 
╔════════════════════════════════════════════════════════════╗
2026-04-26 01:19:21,095 | INFO | ║   CONTRACT REVIEW & REDLINING AI AGENT PIPELINE         ║
2026-04-26 01:19:21,096 | INFO | ║   LegalTech SME Automation | CUAD + LLM Fine-Tuning     ║
2026-04-26 01:19:21,097 | INFO | ╚════════════════════════════════════════════════════════════╝

2026-04-26 01:19:21,103 | INFO | Seed set to 42
2026-04-26 01:19:21,104 | INFO | ============================================================
2026-04-26 01:19:21,104 | INFO | STEP: HuggingFace Authentication
2026-04-26 01:19:21,105 | INFO | ============================================================
2026-04-26 01:19:21,276 | INFO | HTTP Request: GET https://huggingface.co/api/whoami-v2 "HTTP/1.1 200 OK"
2026-04-26 01:19:21,278 | INFO |  HuggingFace login successful (token from /kaggle/input/datasets/tobimichigan/hf-tokens/hf_tokens/hf.token.txt)
2026-04-26 01:19:21,279 | INFO | [MEM start] Process=0.95GB | Avail

  Parsing CUADv1.json:   5%|▍         | 25/510 [00:00<00:00, 2578.70contract/s]


2026-04-26 01:19:22,763 | INFO | [MEM after_load] Process=1.12GB | Available=29.5GB / 31.4GB
2026-04-26 01:19:22,764 | INFO |    Loaded 1,000 QA records from CUADv1.json
2026-04-26 01:19:22,774 | INFO | [MEM after_cuad] Process=1.11GB | Available=29.5GB / 31.4GB
2026-04-26 01:19:23,014 | INFO | Loading CUAD JSON: /kaggle/input/datasets/tobimichigan/cuad-contract-understanding-atticus-dataset/test.json
2026-04-26 01:19:23,018 | INFO | [MEM before_load] Process=1.10GB | Available=29.5GB / 31.4GB
2026-04-26 01:19:23,185 | INFO |   Total contracts in file: 102


  Parsing test.json:  25%|██▍       | 25/102 [00:00<00:00, 2523.16contract/s]


2026-04-26 01:19:23,440 | INFO | [MEM after_load] Process=1.12GB | Available=29.5GB / 31.4GB
2026-04-26 01:19:23,441 | INFO |    Loaded 1,000 QA records from test.json
2026-04-26 01:19:23,445 | INFO | [MEM after_test] Process=1.12GB | Available=29.5GB / 31.4GB
2026-04-26 01:19:23,685 | INFO | Loading CUAD JSON: /kaggle/input/datasets/tobimichigan/cuad-contract-understanding-atticus-dataset/train_separate_questions.json
2026-04-26 01:19:23,690 | INFO | [MEM before_load] Process=1.12GB | Available=29.5GB / 31.4GB
2026-04-26 01:19:24,336 | INFO |   Total contracts in file: 408


...

2026-04-26 01:36:02,296 | INFO | HTTP Request: HEAD https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct/resolve/main/generation_config.json "HTTP/1.1 200 OK"
2026-04-26 01:36:02,352 | INFO | HTTP Request: GET https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct/resolve/main/generation_config.json "HTTP/1.1 200 OK"


generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

2026-04-26 01:36:02,422 | INFO | HTTP Request: HEAD https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct/resolve/main/custom_generate/generate.py "HTTP/1.1 404 Not Found"
2026-04-26 01:36:02,425 | INFO |    Model loaded: meta-llama/Llama-3.1-8B-Instruct
2026-04-26 01:36:03,257 | WARNING |   Could not load fine-tuned model: Error(s) in loading state_dict for PeftModelForCausalLM:
	size mismatch for base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight: copying a param with shape torch.Size([8, 3584]) from checkpoint, the shape in current model is torch.Size([8, 4096]).
	size mismatch for base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight: copying a param with shape torch.Size([3584, 8]) from checkpoint, the shape in current model is torch.Size([4096, 8]).
	size mismatch for base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight: copying a param with shape torch.Size([8, 3584]) from checkpoint, the shape in current model is torch.Si


  Extracting clauses:   0%|          | 0/10 [00:00<?, ?it/s]
                                                            

2026-04-26 01:36:03,277 | INFO |   Extracted 6 clause snippets
2026-04-26 01:36:03,279 | INFO | 
────────────────────────────────────────────────────────────
2026-04-26 01:36:03,280 | INFO |   CONTRACT REVIEW REPORT: Sample NDA 2025
2026-04-26 01:36:03,281 | INFO | ────────────────────────────────────────────────────────────
2026-04-26 01:36:03,282 | INFO |   Issues: 1 total | HIGH=1 | MED=0 | LOW=0
2026-04-26 01:36:03,282 | INFO | 
  REDLINES:
2026-04-26 01:36:03,283 | INFO |     [HIGH] IP Ownership Assignment: ⚠ IP Ownership Assignment clause is MISSING. Standard practice requires: Client …
2026-04-26 01:36:03,284 | INFO | 
  CHECKLIST PREVIEW:
2026-04-26 01:36:03,285 | INFO |     ============================================================
2026-04-26 01:36:03,286 | INFO |     NEGOTIATION CHECKLIST
2026-04-26 01:36:03,287 | INFO |     ============================================================
2026-04-26 01:36:03,287 | INFO |     Generated: 2026-04-26 01:36
2026-04-26 01:36:03,289 |

In [3]:
import os
import zipfile
from tqdm import tqdm
from pathlib import Path
import time

def get_all_files(directory):
    """Recursively get all files in directory and subdirectories"""
    all_files = []
    for root, dirs, files in os.walk(directory):
        for file in files:
            full_path = os.path.join(root, file)
            all_files.append(full_path)
    return all_files

def get_total_size(file_paths):
    """Calculate total size of all files"""
    total_size = 0
    for file_path in file_paths:
        try:
            total_size += os.path.getsize(file_path)
        except (OSError, FileNotFoundError):
            continue
    return total_size

def create_kaggle_working_zip(source_dir="/kaggle/working/", output_name="CONTRACT_REVIEW_&_REDLINING_AI+AGENT.zip"):
    """
    Create a zip file of all content in the Kaggle working directory
    
    Args:
        source_dir (str): Source directory to zip (default: /kaggle/working/)
        output_name (str): Name of the output zip file
    """
    
    # Check if source directory exists
    if not os.path.exists(source_dir):
        print(f"Error: Source directory '{source_dir}' does not exist!")
        return False
    
    # Get all files recursively
    print("Scanning files...")
    all_files = get_all_files(source_dir)
    
    if not all_files:
        print(f"No files found in '{source_dir}'")
        return False
    
    print(f"Found {len(all_files)} files to compress")
    
    # Calculate total size for progress tracking
    total_size = get_total_size(all_files)
    print(f"Total size: {total_size / (1024*1024):.2f} MB")
    
    # Create zip file with progress bar
    try:
        with zipfile.ZipFile(output_name, 'w', zipfile.ZIP_DEFLATED, compresslevel=6) as zipf:
            # Progress bar based on file count
            with tqdm(total=len(all_files), desc="Compressing files", unit="files") as pbar:
                processed_size = 0
                
                for file_path in all_files:
                    try:
                        # Get relative path for the zip archive
                        arcname = os.path.relpath(file_path, source_dir)
                        
                        # Add file to zip
                        zipf.write(file_path, arcname)
                        
                        # Update progress
                        file_size = os.path.getsize(file_path)
                        processed_size += file_size
                        
                        # Update progress bar with file info
                        pbar.set_postfix({
                            'Current': os.path.basename(file_path)[:20],
                            'Size': f"{processed_size / (1024*1024):.1f}MB"
                        })
                        pbar.update(1)
                        
                    except Exception as e:
                        print(f"Warning: Could not add {file_path} to zip: {str(e)}")
                        pbar.update(1)
                        continue
        
        # Get final zip file size
        zip_size = os.path.getsize(output_name)
        compression_ratio = (1 - zip_size / total_size) * 100 if total_size > 0 else 0
        
        print(f"\n Successfully created '{output_name}'")
        print(f" Original size: {total_size / (1024*1024):.2f} MB")
        print(f" Compressed size: {zip_size / (1024*1024):.2f} MB")
        print(f" Compression ratio: {compression_ratio:.1f}%")
        
        return True
        
    except Exception as e:
        print(f"Error creating zip file: {str(e)}")
        return False

def download_zip_in_kaggle(zip_filename):
    """
    Trigger download in Kaggle notebook environment
    """
    try:
        # In Kaggle, files in the working directory are automatically available for download
        # We can also use the files.download() method if available
        from google.colab import files
        files.download(zip_filename)
        print(f"Download triggered for {zip_filename}")
    except ImportError:
        # If not in Colab/Kaggle environment with files API
        print(f"Zip file '{zip_filename}' created successfully!")
        print("In Kaggle, you can download it from the 'Output' tab or use the file browser.")
        print("The file is located in your current working directory.")

if __name__ == "__main__":
    # Configuration
    SOURCE_DIRECTORY = "/kaggle/working/"
    OUTPUT_ZIP_NAME = "CONTRACT_REVIEW_&_REDLINING_AI+AGENT.zip"
    
    print(" Starting Kaggle Working Directory Backup")
    print("=" * 50)
    
    # Create the zip file
    success = create_kaggle_working_zip(SOURCE_DIRECTORY, OUTPUT_ZIP_NAME)
    
    if success:
        print(f"\n Preparing download...")
        download_zip_in_kaggle(OUTPUT_ZIP_NAME)
    else:
        print(" Backup failed!")

 Starting Kaggle Working Directory Backup
Scanning files...
Found 46 files to compress
Total size: 553.02 MB



Compressing files: 100%|██████████| 46/46 [00:29<00:00,  1.55files/s, Current=11_model_comparison_, Size=553.0MB]


 Successfully created 'CONTRACT_REVIEW_&_REDLINING_AI+AGENT.zip'
 Original size: 553.02 MB
 Compressed size: 500.46 MB
 Compression ratio: 9.5%

 Preparing download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download triggered for CONTRACT_REVIEW_&_REDLINING_AI+AGENT.zip
